Url Connection & Data Cleaning

In [147]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv
import openmeteo_requests
import requests_cache
from urllib3.util import Retry
from requests.adapters import HTTPAdapter

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 5

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
    master_df = pd.concat(dataframe_list, ignore_index=True)
    master_df = master_df.drop_duplicates(subset=['latitude', 'longitude', 'bright_ti4', 'acq_date', 'acq_time'])
    print("\nHigh-capacity local aggregation complete!")
    print(f"Combined Dataset Matrix Size: {len(master_df)} Unique Operational Fire Observations")
else:
    print("\nDataset compilation failed. Check your network or API permissions.")
    raise SystemExit

Successfully added 95 fire points from Amazon Rainforest.
Successfully added 208 fire points from California.
Successfully added 2843 fire points from Siberian Taiga.
Successfully added 279 fire points from New South Wales.
Successfully added 1493 fire points from Pantanal.
Successfully added 138 fire points from Alberta.
Successfully added 2852 fire points from Mediterranean Basin.
Successfully added 1606 fire points from Indonesia.
Successfully added 141 fire points from Greece.
Successfully added 1428 fire points from Algeria.

High-capacity local aggregation complete!
Combined Dataset Matrix Size: 9879 Unique Operational Fire Observations


Data Cleaning

In [148]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [149]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                          frp             bright_ti5             bright_ti4  \
                         mean        std        mean        std        mean   
location_name                                                                 
Alberta              6.839928  13.064263  289.597029  11.101858  326.322899   
Algeria              2.744329   3.791625  292.418932   9.261658  317.222986   
Amazon Rainforest    4.683158   2.556835  290.462105   3.882397  334.601895   
California           4.223558  10.402042  292.526346  12.137486  315.605385   
Indonesia            6.704633  12.372719  291.431806   7.148264  328.377055   
Mediterranean Basin  3.900806   8.759803  294.143065   9.890317  319.754208   
New South Wales      3.624731   5.146811  283.077097   5.453960  316.164803   
Pantanal             3.937904   5.861795  291.612458   5.455672  321.446731   
Siberian Taiga       7.808931  12.827919  284.695315  10.870212  325.797154   

                                
                

Creating Risk Level Column

In [150]:
def assign_risk_level(frp_value):
  if frp_value < 5.0:
    return 0
  elif frp_value < 15.0:
    return 1
  elif frp_value < 40.0:
    return 2
  else:
    return 3

master_df['risk_level'] = master_df['frp'].apply(assign_risk_level)

master_df['is_daytime'] = master_df['daynight'].map({'D': 1, 'N': 0})

if master_df['confidence'].dtype == 'O':
    master_df['confidence_num'] = master_df['confidence'].map({'low': 0, 'nominal': 1, 'high': 2})
else:
    master_df['confidence_num'] = master_df['confidence']

# Feature Engineering
master_df['pixel_area'] = master_df['scan'] * master_df['track']
master_df['frp_density'] = master_df['frp'] / master_df['pixel_area']
master_df['thermal_diff'] = master_df['bright_ti4'] / master_df['bright_ti5']
master_df['thermal_ratio'] = master_df['bright_ti4'] / master_df['bright_ti5']
master_df['is_daytime'] = master_df['is_daytime'].fillna(0).astype(int)
master_df['confidence_num'] = pd.to_numeric(master_df['confidence_num'], errors='coerce').fillna(0).astype(float)


# Open Meteo Data
import requests
import time

def fetch_meteo_chunk(lats, lons):
    url = "https://archive-api.open-meteo.com/v1/archive"
    payload = {
        "latitude": lats,
        "longitude": lons,
        "start_date": "2026-05-28",
        "end_date": "2026-06-01",
        "hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "wind_speed_10m"],
        "timezone": "auto",
        "format": "json" 
    }
    
    response = requests.get(url, params=payload)
    if response.status_code == 200:
        return response.json()
    else:
        response.raise_for_status()

SUB_CHUNK_SIZE = 30
all_results = []

for i in range(0, len(master_df), SUB_CHUNK_SIZE):
    chunk_df = master_df.iloc[i : i + SUB_CHUNK_SIZE]
    
    try:
        data = fetch_meteo_chunk(chunk_df['latitude'].tolist(), chunk_df['longitude'].tolist())
        all_results.append(data)
        time.sleep(1.0) 
    except Exception as e:
        print(f"Error processing index range [{i}:{i+SUB_CHUNK_SIZE}]: {e}")

Error processing index range [5010:5040]: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=55.5215&latitude=54.95132&latitude=54.95218&latitude=54.95598&latitude=55.18501&latitude=55.1875&latitude=55.18957&latitude=55.51999&latitude=55.59653&latitude=55.18792&latitude=55.20382&latitude=55.62793&latitude=55.6306&latitude=57.00711&latitude=58.96804&latitude=59.42746&latitude=59.4292&latitude=59.43261&latitude=59.4395&latitude=59.44124&latitude=59.44464&latitude=55.18763&latitude=55.20415&latitude=57.24115&latitude=59.42461&latitude=59.42474&latitude=59.42485&latitude=59.42498&latitude=59.4298&latitude=59.42991&longitude=-119.30835&longitude=-119.19102&longitude=-119.19101&longitude=-119.18647&longitude=-119.87918&longitude=-119.87801&longitude=-119.87488&longitude=-119.30488&longitude=-119.3389&longitude=-119.87978&longitude=-114.9379&longitude=-111.58157&longitude=-111.58177&longitude=-111.47425&longitude=-112.85456&longitude=-113.42685

Prediction: Fire Radiative Power & Bright Fire Spots

In [151]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

X = master_df[['track', 'scan', 'bright_ti5', 'bright_ti4', 'latitude', 'longitude', 'is_daytime', 'confidence_num', 'pixel_area', 'thermal_diff', 'thermal_ratio']]
y = master_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)

# Printing the shapes
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")


# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=120, criterion="log_loss", random_state=42)
rf_model.fit(X_train, y_train, sample_weight=None)
rf_predictions = rf_model.predict(X_test)

# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=100, random_state = 42, verbose=0)
gb_model.fit(X_train, y_train, sample_weight=None)
gb_predictions = gb_model.predict(X_test)
# Add a new Dataframe to see the results clearly
test_results_df = X_test.copy()
test_results_df['True_Risk_Level (Random Forest)'] = y_test
test_results_df['Predicted_Risk_level (Random Forest)'] = rf_predictions
test_results_df['True_Risk_Level (Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (Gradient Boosting)'] = gb_predictions

test_results_df.head(50)

X_train: (7903, 11), X_test: (1976, 11), y_train: (7903,), y_test: (1976,)


,track,scan,bright_ti5,bright_ti4,latitude,longitude,is_daytime,confidence_num,pixel_area,thermal_diff,thermal_ratio,True_Risk_Level (Random Forest),Predicted_Risk_level (Random Forest),True_Risk_Level (Gradient Boosting),Predicted_Risk_Level (Gradient Boosting)
4818,0.41,0.50,296.63,333.22,-20.25041,-56.08891,1,0.0,0.2050,1.123352,1.123352,0,0,0,1
952,0.40,0.47,274.32,335.23,66.70715,80.44839,1,0.0,0.1880,1.222040,1.222040,2,1,2,1
9023,0.38,0.43,289.53,329.86,4.10961,96.93510,1,0.0,0.1634,1.139295,1.139295,0,0,0,0
5212,0.53,0.60,286.74,299.01,43.36483,12.54548,0,0.0,0.3180,1.042791,1.042791,0,0,0,0
7244,0.42,0.53,285.76,300.10,34.91458,37.94805,0,0.0,0.2226,1.050182,1.050182,0,0,0,0
4447,0.37,0.41,296.92,333.14,-16.61197,-59.62464,1,0.0,0.1517,1.121986,1.121986,0,0,0,0
3881,0.36,0.38,292.61,324.01,-16.70763,-59.63536,0,0.0,0.1368,1.107310,1.107310,0,0,0,0
5613,0.38,0.44,292.47,320.24,30.37929,30.56552,0,0.0,0.1672,1.094950,1.094950,0,0,0,0
6148,0.36,0.39,290.88,309.94,45.45850,10.45792,0,0.0,0.1404,1.065525,1.065525,0,0,0,0
6847,0.40,0.47,294.95,341.57,31.05748,8.03357,0,0.0,0.1880,1.158061,1.158061,0,1,0,1


Accuracy Score: How good were the model's guesses?

In [152]:
from sklearn.metrics import accuracy_score as acc
from sklearn.metrics import precision_score as pre
from sklearn.metrics import recall_score as rc
from sklearn.metrics import f1_score as f1
from sklearn.metrics import classification_report as cl_report
rf_acc_score = acc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'])
rf_pre_score = pre(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_rec_score = rc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_f1_score = f1(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')

gb_acc_score = acc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'])
gb_pre_score = pre( test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_rec_score = rc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_f1_score = f1(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: Random Forest --> {rf_acc_score}\n Precision Score: Random Forest --> {rf_pre_score}\n Recall Score: Random Forest --> {rf_rec_score}\n F1 Score: Random Forest --> {rf_f1_score}")
print("\n")
print(f" Accuracy Score: Gradient Boosting --> {gb_acc_score}\n Precision Score: Gradient Boosting --> {gb_pre_score}\n Recall Score: Gradient Boosting --> {gb_rec_score}\n F1 Score: Gradient Boosting --> {gb_f1_score}")
print(cl_report(
    test_results_df['True_Risk_Level (Gradient Boosting)'], 
    test_results_df['Predicted_Risk_Level (Gradient Boosting)'], 
    target_names=['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)', 'Extreme Risk (3)']
))


 Accuracy Score: Random Forest --> 0.8021255060728745
 Precision Score: Random Forest --> 0.7847567618486032
 Recall Score: Random Forest --> 0.8021255060728745
 F1 Score: Random Forest --> 0.7868983395609922


 Accuracy Score: Gradient Boosting --> 0.805161943319838
 Precision Score: Gradient Boosting --> 0.7909202552428833
 Recall Score: Gradient Boosting --> 0.805161943319838
 F1 Score: Gradient Boosting --> 0.7918203833216585
                   precision    recall  f1-score   support

     Low Risk (0)       0.87      0.93      0.90      1379
Moderate Risk (1)       0.63      0.62      0.62       447
    High Risk (2)       0.54      0.18      0.26       114
 Extreme Risk (3)       0.62      0.44      0.52        36

         accuracy                           0.81      1976
        macro avg       0.66      0.54      0.58      1976
     weighted avg       0.79      0.81      0.79      1976



Prediction: Only 83-85% F1 Score, let's test XG Gradient Boosting

In [153]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
xgb_model = XGBClassifier(objective='multi:softprob',
    num_class=4,
    learning_rate=0.15,     
    max_depth=7,            
    min_child_weight=1,   
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_preds = xgb_model.predict(X_test)
test_results_df['True_Risk_Level (XG Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'] = xgb_preds
xgb_acc_score = acc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'])
xgb_pre_score = pre( test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_rec_score = rc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_f1_score = f1(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: XG Gradient Boosting --> {xgb_acc_score}\n Precision Score: XG Gradient Boosting--> {xgb_pre_score}\n Recall Score: XG Gradient Boosting --> {xgb_rec_score}\n F1 Score: XG Gradient Boosting --> {xgb_f1_score}")
print(classification_report(y_test, xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

# The results weren't as I expected, so I will now try GridSearch to find the best settings
xgb_param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42
)

xgb_grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=1,
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train, sample_weight=sample_weights)
best_xgb = xgb_grid.best_estimator_
print(f"\nWinning XGBoost Settings: {xgb_grid.best_params_}")

tuned_xgb_preds = best_xgb.predict(X_test)
print("\nTUNED XGBOOST SCORECARD:")
print(classification_report(y_test, tuned_xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

 Accuracy Score: XG Gradient Boosting --> 0.7970647773279352
 Precision Score: XG Gradient Boosting--> 0.809476006922326
 Recall Score: XG Gradient Boosting --> 0.7970647773279352
 F1 Score: XG Gradient Boosting --> 0.8015533322097036
              precision    recall  f1-score   support

         Low       0.92      0.87      0.89      1379
    Moderate       0.59      0.70      0.64       447
        High       0.43      0.39      0.41       114
     Extreme       0.50      0.50      0.50        36

    accuracy                           0.80      1976
   macro avg       0.61      0.62      0.61      1976
weighted avg       0.81      0.80      0.80      1976

Fitting 3 folds for each of 36 candidates, totalling 108 fits

Winning XGBoost Settings: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 0.8}

TUNED XGBOOST SCORECARD:
              precision    recall  f1-score   support

         Low       0.94      0.82      0.88      1379
    Moderate       0.53 